### Tool Creation Now

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('LLM_API')
from langchain.chat_models import init_chat_model
model=init_chat_model('groq:qwen/qwen3-32b')

In [2]:
response=model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. Let me start by recalling what I know about parrots and their vocal abilities. First, parrots are known for their ability to mimic human speech, but not all parrots talk. I think this is a common misconception. Maybe I should clarify that not every parrot talks, but some species are better at it than others.\n\nWhy do they mimic sounds? Well, in the wild, parrots use vocalizations for communication, like calling to each other, warning of danger, or attracting mates. Mimicking sounds could help them blend into their environment. For example, if they mimic the calls of other animals, maybe they can avoid predators or communicate with flock members. But how does this translate to talking to humans?\n\nI remember reading that parrots have a specialized vocal organ called the syrinx, which allows them to produce a wide range of sounds. That\'s different from humans, who use vocal cords. The syrinx is located at the ba

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("Whats the weather in Boston")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Tool: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure to format the tool call correctly within the XML tags as instructed.\n', 'tool_calls': [{'id': '8pn5bb5ht', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 151, 'total_tokens': 244, 'completion_time': 0.13457894, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.006214887, 'prompt_tokens_details': None, 'queue_time': 0.055177673, 'total_time': 0.140793827}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'


### Tool Execution Loop

In [6]:
# Step 1: Model generates tool calls
messages = [{'role':'user','content':"What's the weather in boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Exectute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_res = get_weather.invoke(tool_call)
    messages.append(tool_res)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response)

content='The weather in Boston is sunny.' additional_kwargs={'reasoning_content': "Okay, the user asked about the weather in Boston. I called the get_weather function with Boston as the location. The response came back saying it's sunny there. Now I need to relay that information clearly. Let me check if there's any additional details I should mention, but the function only provided the weather condition. So I'll just state that it's sunny in Boston. Keep it simple and direct.\n"} response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 192, 'total_tokens': 286, 'completion_time': 0.157173604, 'completion_tokens_details': {'reasoning_tokens': 82}, 'prompt_time': 0.007737848, 'prompt_tokens_details': None, 'queue_time': 0.054208216, 'total_time': 0.164911452}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e4de1-5163-7bb2-a4f4-ed6b8

In [7]:
messages

[{'role': 'user', 'content': "What's the weather in boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there\'s no typo. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': '0t37ww60b', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 154, 'total_tokens': 247, 'completion_time': 0.145527086, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.006075512, 'prompt_tokens_details': None, 'queue_time': 0.053408352, 'total_time': 0.151602598}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_